# Notebook 02 — DMSP-OLS → VIIRS Nighttime Lights Harmonisation

DMSP-OLS covers 1992–2013 (digital number, DN, 0–63).
VIIRS-DNB covers 2012–present (radiance, nW/cm²/sr).

We use the **Li & Zhou (2017)** calibration:
```
DN_equiv = 10.062 × ln(avg_rad + 1)
```
This notebook validates the bridge using the 2012–2013 overlap year.

Reference: Li, X. & Zhou, Y. (2017). Remote Sensing of Environment.
https://doi.org/10.1016/j.rse.2017.07.037

In [ ]:
import sys; sys.path.insert(0, '..')
import ee, math, pandas as pd, matplotlib.pyplot as plt
ee.Initialize()

# Pull both sensors for the overlap year (2012) at Bangalore
LAT, LON = 12.97, 77.59
region = ee.Geometry.Point([LON, LAT]).buffer(5000)  # 5km for light halo

dmsp_2012 = (
    ee.ImageCollection('NOAA/DMSP-OLS/NIGHTTIME_LIGHTS')
    .filter(ee.Filter.calendarRange(2012, 2012, 'year'))
    .select('stable_lights').mean()
    .reduceRegion(ee.Reducer.mean(), region, 1000).getInfo().get('stable_lights')
)

viirs_2012 = (
    ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG')
    .filter(ee.Filter.calendarRange(2012, 2012, 'year'))
    .select('avg_rad').mean()
    .reduceRegion(ee.Reducer.mean(), region, 500).getInfo().get('avg_rad')
)

viirs_calibrated = 10.062 * math.log(viirs_2012 + 1)

print(f'DMSP 2012:             {dmsp_2012:.2f} DN')
print(f'VIIRS 2012 (raw):      {viirs_2012:.4f} nW/cm²/sr')
print(f'VIIRS calibrated:      {viirs_calibrated:.2f} DN-equiv')
print(f'Bridge error:          {abs(dmsp_2012 - viirs_calibrated):.2f} DN ({abs(dmsp_2012 - viirs_calibrated)/dmsp_2012*100:.1f}%)')

In [ ]:
# Validate across multiple cities
cities = [
    ('Bangalore', 12.97, 77.59),
    ('Lagos', 6.5, 3.4),
    ('Dubai', 25.2, 55.3),
    ('Karlsruhe', 49.0, 8.4),
]
rows = []
for name, lat, lon in cities:
    r = ee.Geometry.Point([lon, lat]).buffer(5000)
    dmsp = ee.ImageCollection('NOAA/DMSP-OLS/NIGHTTIME_LIGHTS').filter(ee.Filter.calendarRange(2013, 2013, 'year')).select('stable_lights').mean().reduceRegion(ee.Reducer.mean(), r, 1000).getInfo().get('stable_lights')
    viirs = ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG').filter(ee.Filter.calendarRange(2013, 2013, 'year')).select('avg_rad').mean().reduceRegion(ee.Reducer.mean(), r, 500).getInfo().get('avg_rad')
    rows.append({'city': name, 'dmsp_dn': dmsp, 'viirs_raw': viirs, 'viirs_calib': 10.062 * math.log((viirs or 0) + 1)})

df = pd.DataFrame(rows)
df['error_pct'] = abs(df.dmsp_dn - df.viirs_calib) / df.dmsp_dn * 100
print(df.to_string(index=False))